# 🫀 Heart Failure Prediction Using Machine Learning
### **Big Data Analysis & Machine Learning Laboratory**
**Student Name:** K LAKSHMI PRASANNA  
**Roll Number:** 24691A32I9  
**Department:** Computer Science & Engineering - Data Science  
**Institution:** Madanapalle Institute of Technology & Science (MITS) / JNTUA

---
## 📌 1. Project Overview
Cardiovascular diseases (CVDs) are the leading cause of death globally, taking an estimated 17.9 million lives each year. Heart failure is a common event caused by CVDs. Early automated detection and risk stratification through clinical biomarkers can substantially reduce mortality rates.

This notebook implements an end-to-end Machine Learning pipeline that evaluates **7 distinct classifiers**:
1. **Logistic Regression (L2 Regularized)**
2. **Support Vector Classifier (RBF Kernel)**
3. **K-Nearest Neighbors (KNN)**
4. **Random Forest Classifier**
5. **Gradient Boosting Classifier (GBDT)**
6. **Multi-Layer Perceptron (Artificial Neural Network)**
7. **Soft Voting Ensemble Classifier**

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

print('All Machine Learning and Data Science libraries successfully imported.')

## 📊 2. Data Loading & Exploratory Data Analysis (EDA)

In [2]:
df = pd.read_csv('data/heart_failure.csv')
print('Dataset Shape:', df.shape)
df.head()

In [3]:
print('Missing Values Check:')
print(df.isnull().sum())

print('\nStatistical Summary of Clinical Biomarkers:')
df.describe().T

## ⚙️ 3. Data Preprocessing & Stratified Train-Test Splitting

In [4]:
feature_cols = [c for c in df.columns if c not in ['Patient_ID', 'HeartDisease']]
X = df[feature_cols]
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training Samples: {X_train.shape[0]} | Testing Samples: {X_test.shape[0]}')

## 🤖 4. Model Training & 5-Fold Stratified Cross-Validation

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42),
    'Multi-Layer Perceptron': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=700, random_state=42)
}

models['Voting Ensemble'] = VotingClassifier(
    estimators=[('rf', models['Random Forest']), ('gbdt', models['Gradient Boosting']), ('lr', models['Logistic Regression'])],
    voting='soft'
)

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1_Score': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else 0,
        'CV_Mean': cv_scores.mean(),
        'CV_Std': cv_scores.std()
    })

df_metrics = pd.DataFrame(results)
df_metrics.sort_values(by='F1_Score', ascending=False)

## 📈 5. Clinical Feature Importance & Visualizations

In [6]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#2b5c8f')
plt.title('Clinical Feature Importance Ranking (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Gini Importance', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 🩺 6. Real-Time Patient Risk Inference Test

In [7]:
from predict import predict_patient_risk

test_patient = {
    'Age': 65, 'Sex': 1, 'ChestPainType': 3, 'RestingBP': 155, 'Cholesterol': 270,
    'FastingBS': 1, 'RestingECG': 2, 'MaxHR': 110, 'ExerciseAngina': 1,
    'Oldpeak': 2.5, 'ST_Slope': 2, 'EjectionFraction': 30, 'SerumCreatinine': 2.1
}

predict_patient_risk(test_patient)